# Real Data Sanity Dashboard

**Goal**: Verify that all spatial layers actually align to the same Mumbai region.

Panels:
1. DEM (Elevation)
2. Slope
3. Flow Accumulation
4. OSM Roads overlay on DEM
5. Waterways
6. Distance to Water
7. Emergency Assets
8. Combined overlay: DEM + Roads + Waterways

**Pass criterion**: All layers render and show Mumbai region (bbox ~72.75–73.05°E, 18.85–19.30°N)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rasterio
import json

ROOT = Path('../').resolve()
STATIC = ROOT / 'data' / 'processed' / 'static'

# Mumbai bounding box in WGS84
BBOX_WGS84 = [72.75, 18.85, 73.05, 19.30]
print('Paths exist:', {p.name: p.exists() for p in STATIC.glob('*')})

In [ ]:
def read_tif(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        data[data == src.nodata] = np.nan
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    return data, extent

def safe_read(name):
    p = STATIC / name
    if not p.exists():
        print(f'MISSING: {name}')
        return None, None
    return read_tif(p)

In [ ]:
# Load all rasters
elev, elev_ext = safe_read('elevation.tif')
slope, slope_ext = safe_read('slope.tif')
flow, flow_ext = safe_read('flow_accumulation.tif')
dtw, dtw_ext = safe_read('distance_to_water.tif')

# Load vectors
def safe_gdf(name):
    p = STATIC / name
    if not p.exists():
        print(f'MISSING: {name}')
        return None
    return gpd.read_file(str(p))

roads_gdf = safe_gdf('roads.geojson')
water_gdf = safe_gdf('waterways.geojson')
ea_gdf = safe_gdf('emergency_assets.geojson')
print('Data loaded!')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('JalRakshak — Mumbai Real Data Sanity Dashboard', fontsize=16, fontweight='bold')

def panel(ax, data, ext, title, cmap='viridis', log_scale=False, vmin=None, vmax=None):
    if data is None:
        ax.text(0.5, 0.5, f'MISSING\n{title}', ha='center', va='center',
                color='red', fontsize=12, transform=ax.transAxes)
        ax.set_title(title)
        return
    d = np.log1p(data) if log_scale else data
    im = ax.imshow(d, cmap=cmap, extent=ext, origin='upper', vmin=vmin, vmax=vmax,
                   aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='log(1+x)' if log_scale else '')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')

# Panel 1: DEM
panel(axes[0,0], elev, elev_ext, '1. Elevation (m)', cmap='terrain')

# Panel 2: Slope
panel(axes[0,1], slope, slope_ext, '2. Slope (°)', cmap='magma')

# Panel 3: Flow Accumulation
panel(axes[0,2], flow, flow_ext, '3. Flow Accumulation (log scale)',
      cmap='Blues', log_scale=True)

# Panel 4: Roads overlay on DEM
ax4 = axes[0,3]
if elev is not None:
    ax4.imshow(elev, cmap='Greys_r', extent=elev_ext, origin='upper',
               vmin=np.nanpercentile(elev, 2), vmax=np.nanpercentile(elev, 98),
               aspect='auto')
if roads_gdf is not None and len(roads_gdf) > 0:
    roads_proj = roads_gdf.to_crs('EPSG:32643')
    roads_proj.plot(ax=ax4, color='orangered', linewidth=0.4, alpha=0.6)
ax4.set_title('4. OSM Roads on DEM', fontsize=11, fontweight='bold')
ax4.set_xlabel('Easting (m)')

# Panel 5: Waterways
ax5 = axes[1,0]
if elev is not None:
    ax5.imshow(elev, cmap='Greys_r', extent=elev_ext, origin='upper',
               vmin=np.nanpercentile(elev, 2), vmax=np.nanpercentile(elev, 98),
               aspect='auto')
if water_gdf is not None and len(water_gdf) > 0:
    water_proj = water_gdf.to_crs('EPSG:32643')
    water_proj.plot(ax=ax5, color='dodgerblue', linewidth=1.0, alpha=0.8)
ax5.set_title('5. Waterways', fontsize=11, fontweight='bold')
ax5.set_xlabel('Easting (m)')

# Panel 6: Distance to Water
panel(axes[1,1], dtw, dtw_ext, '6. Distance to Water (m)', cmap='YlOrRd_r')

# Panel 7: Emergency Assets
ax7 = axes[1,2]
if elev is not None:
    ax7.imshow(elev, cmap='Greys_r', extent=elev_ext, origin='upper',
               vmin=np.nanpercentile(elev, 2), vmax=np.nanpercentile(elev, 98),
               aspect='auto')
if ea_gdf is not None and len(ea_gdf) > 0:
    ea_proj = ea_gdf.to_crs('EPSG:32643')
    colors_by_cat = {'hospitals': 'red', 'schools': 'gold',
                     'fire_stations': 'orange', 'police_stations': 'blue',
                     'shelters': 'green'}
    for cat, color in colors_by_cat.items():
        subset = ea_proj[ea_proj.get('category', '') == cat]
        if len(subset) > 0:
            subset_pts = subset[subset.geometry.geom_type == 'Point']
            if len(subset_pts) > 0:
                ax7.scatter(subset_pts.geometry.x, subset_pts.geometry.y,
                            c=color, s=30, label=cat, zorder=5)
    ax7.legend(loc='upper right', fontsize=6)
ax7.set_title('7. Emergency Assets', fontsize=11, fontweight='bold')
ax7.set_xlabel('Easting (m)')

# Panel 8: Combined overlay
ax8 = axes[1,3]
if elev is not None:
    ax8.imshow(elev, cmap='terrain', extent=elev_ext, origin='upper',
               vmin=np.nanpercentile(elev, 2), vmax=np.nanpercentile(elev, 98),
               aspect='auto', alpha=0.7)
if roads_gdf is not None:
    roads_gdf.to_crs('EPSG:32643').plot(ax=ax8, color='gray', linewidth=0.3, alpha=0.5)
if water_gdf is not None:
    water_gdf.to_crs('EPSG:32643').plot(ax=ax8, color='dodgerblue', linewidth=1.0)
ax8.set_title('8. Combined: DEM + Roads + Water', fontsize=11, fontweight='bold')
ax8.set_xlabel('Easting (m)')

plt.tight_layout()
plt.savefig(str(ROOT / 'reports' / 'sanity_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Sanity dashboard rendered. Saved to reports/sanity_dashboard.png')

In [ ]:
# Print alignment summary
print('=== Alignment Check ===')
layers_ok = True
for name, data, ext in [
    ('Elevation', elev, elev_ext),
    ('Slope', slope, slope_ext),
    ('Flow Acc.', flow, flow_ext),
    ('Dist Water', dtw, dtw_ext),
]:
    if data is None:
        print(f'  ❌ {name}: MISSING')
        layers_ok = False
    else:
        print(f'  ✅ {name}: shape={data.shape}, extent=[{ext[0]:.0f}, {ext[1]:.0f}, {ext[2]:.0f}, {ext[3]:.0f}]')

for name, gdf in [('Roads', roads_gdf), ('Waterways', water_gdf), ('Emergency', ea_gdf)]:
    if gdf is None:
        print(f'  ❌ {name}: MISSING')
        layers_ok = False
    else:
        print(f'  ✅ {name}: {len(gdf)} features, CRS={gdf.crs}')

if layers_ok:
    print('\n✅ SANITY CHECK PASSED: All layers present and aligned to Mumbai.')
else:
    print('\n⚠️  Some layers missing — run the relevant pipelines first.')